# Hadith Translation — Qwen2.5-7B on Colab T4

Translates one book per session. Checkpoints every 50 hadiths so you can resume across sessions.

**Before running:** Upload `{book}.json` and `{book}_narrators.json` from `colab_translate/splits/`.
To resume a previous session, also upload `checkpoint_{book}.json`.

In [ ]:
# === CONFIG — change BOOK each session ===
BOOK = "bukhari"  # one of: bukhari, muslim, abudawud, nasai, tirmidhi, ibnmajah
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct-AWQ"
CHECKPOINT_EVERY = 50

!pip install -q transformers autoawq torch

In [ ]:
import json
import os
import time
from google.colab import files

# Upload files
print(f"Upload {BOOK}.json and {BOOK}_narrators.json")
print(f"Optionally upload checkpoint_{BOOK}.json to resume")
uploaded = files.upload()

# Load hadiths
with open(f"{BOOK}.json", encoding="utf-8") as f:
    hadiths = json.load(f)
print(f"Loaded {len(hadiths)} hadiths")

# Load narrators
with open(f"{BOOK}_narrators.json", encoding="utf-8") as f:
    narrators = json.load(f)
print(f"Loaded {len(narrators)} narrators")

# Load checkpoint if present
checkpoint_file = f"checkpoint_{BOOK}.json"
checkpoint = None
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, encoding="utf-8") as f:
        checkpoint = json.load(f)
    print(f"Resuming from checkpoint: {checkpoint['last_index'] + 1}/{len(hadiths)} hadiths done")
    if checkpoint.get("narrators_done"):
        print("Narrators already translated")
else:
    print("No checkpoint — starting fresh")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype="auto",
)
print("Model loaded")


def generate(prompt: str, max_tokens: int = 512) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
    )
    # Decode only the new tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def fix_honorifics(text: str) -> str:
    for phrase in ["PBUH", "pbuh", "peace be upon him", "Peace be upon him", "Peace Be Upon Him"]:
        text = text.replace(phrase, "\uFDFA")
    return text

In [ ]:
# === Translate narrator names ===
# Skip if already done in a previous session
if checkpoint and checkpoint.get("narrators_done"):
    translated_narrators = checkpoint["narrators"]
    print(f"Narrators already translated ({len(translated_narrators)})")
else:
    translated_narrators = []
    batch_size = 20
    total = len(narrators)
    print(f"Translating {total} narrators in batches of {batch_size}...")

    for i in range(0, total, batch_size):
        batch = narrators[i:i + batch_size]
        numbered = "\n".join(
            f"{j+1}. {n['name_ar']}" for j, n in enumerate(batch)
        )
        prompt = (
            "Transliterate these Arabic names to English. "
            "Output ONLY the English names, one per line, numbered to match. "
            f"No explanations.\n\n{numbered}"
        )
        response = generate(prompt, max_tokens=256)
        lines = [l.strip() for l in response.strip().split("\n") if l.strip()]

        for j, n in enumerate(batch):
            name_en = ""
            if j < len(lines):
                # Strip numbering like "1. " or "1) "
                name_en = lines[j].lstrip("0123456789.) ").strip()
                name_en = fix_honorifics(name_en)
                if len(name_en) > 200:
                    name_en = ""
            translated_narrators.append({
                "name_ar": n["name_ar"],
                "name_en": name_en,
            })

        done = min(i + batch_size, total)
        print(f"  {done}/{total} narrators", end="\r")

    print(f"\nDone: {len(translated_narrators)} narrators translated")

# Save narrators
with open(f"translated_{BOOK}_narrators.json", "w", encoding="utf-8") as f:
    json.dump(translated_narrators, f, ensure_ascii=False, indent=1)
print(f"Saved translated_{BOOK}_narrators.json")

In [ ]:
# === Translate hadiths with checkpointing ===
from tqdm.auto import tqdm

# Resume state
if checkpoint and "hadiths" in checkpoint:
    translated_hadiths = checkpoint["hadiths"]
    start_index = checkpoint["last_index"] + 1
else:
    translated_hadiths = []
    start_index = 0

total = len(hadiths)
print(f"Translating hadiths {start_index}..{total-1} ({total - start_index} remaining)")


def save_checkpoint(idx):
    ckpt = {
        "book": BOOK,
        "last_index": idx,
        "narrators_done": True,
        "narrators": translated_narrators,
        "hadiths": translated_hadiths,
    }
    with open(checkpoint_file, "w", encoding="utf-8") as f:
        json.dump(ckpt, f, ensure_ascii=False)


pbar = tqdm(range(start_index, total), initial=start_index, total=total)
for idx in pbar:
    h = hadiths[idx]
    arabic = h.get("matn") or h.get("text_ar", "")
    if not arabic:
        translated_hadiths.append({
            "hadith_number": h["hadith_number"],
            "text_ar": h.get("text_ar", ""),
            "text_en": "",
        })
        continue

    prompt = (
        "Translate this Islamic hadith from Arabic to English. "
        "Use the symbol \uFDFA after mentions of the Prophet Muhammad. "
        "Never write PBUH or 'peace be upon him'. "
        f"Output ONLY the English translation.\n\n{arabic}"
    )
    english = generate(prompt)
    english = fix_honorifics(english)

    translated_hadiths.append({
        "hadith_number": h["hadith_number"],
        "text_ar": h.get("text_ar", ""),
        "text_en": english,
    })

    # Checkpoint every N hadiths
    if (idx + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(idx)
        pbar.set_postfix(saved=f"ckpt@{idx+1}")

# Final save
save_checkpoint(total - 1)
print(f"\nDone: {len(translated_hadiths)} hadiths translated")

# Save final output
with open(f"translated_{BOOK}.json", "w", encoding="utf-8") as f:
    json.dump(translated_hadiths, f, ensure_ascii=False, indent=1)
print(f"Saved translated_{BOOK}.json")

In [ ]:
# === Download results ===
from google.colab import files

print("Downloading translated files + checkpoint...")
files.download(f"translated_{BOOK}.json")
files.download(f"translated_{BOOK}_narrators.json")
files.download(f"checkpoint_{BOOK}.json")
print("Done! Save these files to colab_translate/translations/")